In [12]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import seaborn as sns
from pathlib import Path
import plotly.graph_objects as go
pio.renderers.default = "notebook_connected"

In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.base_prep import load_time_indexed_csv, ensure_clean_datetime_index, missing_summary
from src.time_features import add_calendar_features, get_calendar_feature_columns
from src.lag_features import add_lag_features, add_rolling_features, add_trend_features
from src.dataset_builder import build_multiple_target_datasets, save_target_datasets_parquet

In [14]:
INPUT_PATH = r"C:\Data_analysis\Thesis\Data\02_Preprocessing\LF_data_droped.csv"
SAVE_DIR = Path(r"C:\Data_analysis\Thesis\Data\03_Training\Data_with_NaNs\5_min")  # chnage path when resolution is Diff

In [15]:
df = load_time_indexed_csv(INPUT_PATH)
df = ensure_clean_datetime_index(df)

print(df.index.dtype)
print(df.shape)

datetime64[us]
(33696, 9)


In [16]:
ms = missing_summary(df)
print(ms[ms["missing_count"] > 0])

n_rows_with_missing = df.isnull().any(axis=1).sum()
print(f"Rows with any missing values: {n_rows_with_missing}")

                                missing_count  missing_pct
BU_TotActPwr_SDB_EL_Substation           7121        21.13
BA_Soc                                   2551         7.57
PV_WS_Radiation                          1399         4.15
PV_WS_AirTemp                            1397         4.15
PV_WS_RelHum                             1382         4.10
BU_TotActPwr_Tech_Room                    360         1.07
BU_TotActPwr_Academy                      357         1.06
BA_TotActPwr_BESS_AC_Panel1               334         0.99
BA_TotActPwr_BESS_AC_Panel2               331         0.98
Rows with any missing values: 9428


In [17]:
df = add_calendar_features(df, include_extended=True)
calendar_cols = get_calendar_feature_columns(include_extended=True)

In [ ]:
target_cols = [
    "BU_TotActPwr_Academy",
    "BA_TotActPwr_BESS_AC_Panel1",
    "BA_TotActPwr_BESS_AC_Panel2",
    "BU_TotActPwr_Tech_Room",
    "BU_TotActPwr_SDB_EL_Substation",
]

support_cols = [
    "BA_Soc",
    "PV_WS_AirTemp",
    "PV_WS_Radiation",
    "PV_WS_RelHum",
]
lags = [1, 2, 3, 6, 12, 24, 36, 48, 72, 144, 288, 576]
rolling_windows = [6, 12, 36, 288]
std_windows = [12, 36]
trend_pairs = [(1, 12), (1, 288)]

In [19]:
df = add_lag_features(df, target_cols, lags=lags)

df = add_rolling_features(
    df,
    target_cols,
    rolling_windows=rolling_windows,
    add_std_windows=std_windows,
)

df = add_trend_features(
    df,
    target_cols,
    trend_pairs=trend_pairs,
)

In [20]:
datasets = build_multiple_target_datasets(
    df,
    target_cols=target_cols,
    support_cols=support_cols,
    calendar_cols=calendar_cols,
    include_calendar=True,
    include_lags=True,
    include_rolls=True,
    include_trends=True,
    drop_startup=True,
    startup_rows=576,
    drop_target_nan=False,
)

In [21]:
for name, dfx in datasets.items():
    print(name, dfx.shape)

BU_TotActPwr_SDB_EL_Substation (33120, 39)


In [22]:
saved_files = save_target_datasets_parquet(datasets, SAVE_DIR)

for k, v in saved_files.items():
    print(k, v, v.exists())

BU_TotActPwr_SDB_EL_Substation C:\Data_analysis\Thesis\Data\03_Training\Data_with_NaNs\5_min\df_BU_TotActPwr_SDB_EL_Substation.parquet True
